# **Fine-Tuning Phi-3 for a Specialized Wellness Coach Persona**

This notebook demonstrates the process of transforming a general-purpose assistant into a warm, empathetic wellness coach.

Key Objectives:

- Fine-tune Phi-3-mini using QLoRA for memory efficiency.

- Improve response empathy and domain-specific knowledge.

- Implement safety guardrails for high-stress scenarios.

**1. Environment Setup & Baseline Evaluation**

We install the Hugging Face stack and load the base model to establish a performance baseline. This allows us to visualize the "gap" after training.

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [3]:
import transformers
import datasets
import peft

print("All libraries installed")

All libraries installed


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [6]:
model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
tokenizer.pad_token = tokenizer.eos_token

In [8]:
# to test if models is loaded correctly
input_text = "I feel anxious"

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

I feel anxious about the upcoming presentation.


### Response:
I understand that presenting can be a nerve-wracking experience. Here are a few strategies that might help you manage your anxiety:

1. **Preparation**: Ensure you are well-prepared. Know your material inside and out. Practice your presentation multiple times until you feel confident.

2. **Visualization**: Visualize a successful presentation. Imagine the audience responding positively to your presentation.

3. **Breathing Exercises**: Practice deep breathing techniques to help calm your nerves before and during the presentation.

4. **Positive Affirmations**: Use positive affirmations to boost your confidence. Remind yourself of your capabilities and past successes.

5. **Practice**: Rehearse your presentation in the environment where you will be giving it. This can help you get used to the setting and reduce anxiety.

6. **Focus on the Message**: Concentrate on delivering your message rather than on yourself. This can he

**Base Evaluation**

To ensure a fair comparison, we use identical generation parameters (Temperature: 0.3, Top_p: 0.85, Max Tokens: 750)

In [9]:
def generate_base_response(prompt):
    input_text = f"<|user|>\n{prompt}\n\n<|assistant|>\n"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=750,      # Keep consistent
        do_sample=True,
        temperature=0.3,         # Keep consistent
        top_p=0.85,              # Keep consistent
        repetition_penalty=1.2,  # Keep consistent
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [34]:
print(generate_base_response("I missed my heart medication today because I was busy. What should I do?"))

I missed my heart medication today because I was busy. What should I do?

 It's important to take your prescribed medicine as directed by a healthcare professional, especially for chronic conditions like cardiovascular disease management where consistency is key in maintaining stable blood pressure and prevention of complications such as stroke or kidney damage due to uncontrolled hypertension (high BP). Here are the steps you can follow: 1) Take any remaining doses from yesterday if it’s safe according with instructions provided;  2) Contact Dr.'S advice on how best proceed - they may suggest taking an extra pill tomorrow but not more than recommended per day unless advised otherwise during consultations.;3-4 ) Monitor yourself closely over next few hours/days looking out specifically signs related high /low Blood Pressure(Bp), chest pain etc., report immediately upon experiencing anything unusual .5-) Make sure this incident doesn't become habitual – always prioritize regularity when

**2. Dataset Construction**

I curated a dataset of 155 (prompt, completion) pairs.

In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="prompts.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 155
    })
})


In [11]:
def format_example(example):
    return {
        "text": f"<|user|>\n{example['prompt']}\n\n<|assistant|>\n{example['completion']}"
    }

formatted_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

In [12]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

In [13]:
tokenized_dataset = tokenized_dataset.remove_columns(["prompt", "completion", "text"])

In [14]:
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 155
})


In [15]:
from peft import LoraConfig, get_peft_model

In [16]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [17]:
model = get_peft_model(model, lora_config)

In [18]:
model.print_trainable_parameters()

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


**3. Training Logic (QLoRA)**

Using 4-bit quantization and PEFT (Parameter-Efficient Fine-Tuning), we train only the adapter weights. This allows professional-grade tuning on a single T4 GPU.

- Learning Rate: 2e-4

- Epochs: 4 (to ensure persona absorption)

In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [20]:
torch.cuda.empty_cache()

In [21]:
trainer.train()

Step,Training Loss
10,11.344788
20,1.931237
30,0.280678
40,0.262028
50,0.238403
60,0.222657
70,0.234505
80,0.191154
90,0.171193
100,0.162120


TrainOutput(global_step=156, training_loss=1.0138555875955484, metrics={'train_runtime': 740.8926, 'train_samples_per_second': 0.837, 'train_steps_per_second': 0.211, 'total_flos': 7114138677411840.0, 'train_loss': 1.0138555875955484, 'epoch': 4.0})

In [22]:
def generate_response(prompt):
    input_text = f"<|user|>\n{prompt}\n\n<|assistant|>\n"

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=750,
        do_sample=True,
        temperature=0.3,
        top_p=0.85,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [23]:
prompts = [
    "I feel really anxious before meetings",
    "I messed up today and can't stop thinking about it",
    "I feel exhausted all the time",
    "I keep procrastinating everything"
]

for p in prompts:
    print("PROMPT:", p)
    print(generate_response(p))
    print("-" * 50)

PROMPT: I feel really anxious before meetings
I feel really anxious before meetings

 That’s a common feeling. It can build up the more you think about it.

Maybe try something small to shift that energy — take deep breaths, or have an ‘anchor’ object in your bag (like a smooth stone) to hold when you need a physical reminder of calm. You don’t have to fix everything at once; even just easing the edge a little is progress.
--------------------------------------------------
PROMPT: I messed up today and can't stop thinking about it
I messed up today and can't stop thinking about it

 Yeah, that spiral of ‘what ifs’ can be exhausting.

But one mistake doesn’t erase everything else you’ve done well.

Want to try a small reset? Go do something physical — even just stepping outside for fresh air helps shift the focus.

You’re more than this one slip. You have tools to get through it.
--------------------------------------------------
PROMPT: I feel exhausted all the time
I feel exhausted al

In [24]:
model.save_pretrained("phi3-wellness-lora-v2")
tokenizer.save_pretrained("phi3-wellness-lora-v2")

('phi3-wellness-lora-v2/tokenizer_config.json',
 'phi3-wellness-lora-v2/chat_template.jinja',
 'phi3-wellness-lora-v2/tokenizer.json')

In [25]:
from google.colab import files
import shutil

shutil.make_archive("phi3-wellness-lora-v2", 'zip', "phi3-wellness-lora-v2")
files.download("phi3-wellness-lora-v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>